# SH .stl generation

Packages import

In [23]:
import numpy as np
import scipy.special as spf
from stl import mesh  # numpy-stl
import os 

Set folders where raw and optimized `.stl` files will be stored.

In [24]:
SH_DIR = 'SH'
SH_OPT_DIR = 'SH_optimized'

Define functions for `.stl` generation.
`spfunc()` can be redefined in order to draw any $r(\theta, \phi)$ surface.

In [76]:
def spfunc(phi, theta, m, l, r):
    """Points on real spherical harmonic (SH) surface.
    m, l are integers, l>=0, l>=|m|.
    r is a radius of little supporting sphere in a middle of SH."""
    # Real SH [doi.org/10.1016/S0166-1280(97)00185-1]
    Phi, Theta = np.meshgrid(phi, theta)
    
    if m > 0:
        HARMONIC = np.sqrt(2) * (-1)**m * spf.sph_harm(abs(m), l, Phi, Theta).real
    elif m < 0:
        HARMONIC = np.sqrt(2) * (-1)**m * spf.sph_harm(abs(m), l, Phi, Theta).imag
    else:
        HARMONIC = spf.sph_harm(abs(m), l, Phi, Theta).real

    ABS = np.abs(HARMONIC) * 100  # useful scale
    # add little sphere inside SH
    R = np.where(ABS > r, ABS, r)
    X = R * np.sin(Theta) * np.cos(Phi)
    Y = R * np.sin(Theta) * np.sin(Phi)
    Z = R * np.cos(Theta)
    
    return np.stack((X, Y, Z), axis=2).reshape(Phi.shape[0]*Phi.shape[1],3)


def linrange(start, step, num):
    stop = start + step*num
    return np.linspace(start, start + step*num, num, endpoint=False)


def faces_generator(phi, theta):
    """Undocumented.
    Returns mesh faces for stl."""
    start_equator_triangles = np.array([[len(phi),2*len(phi),2*len(phi)+1],[len(phi),2*len(phi)+1,len(phi)+1]])
    equator_triangles = linrange(linrange(start_equator_triangles, 1, len(phi)-1), len(phi), len(theta)-3)
    equator_faces = equator_triangles.reshape(np.prod(equator_triangles.shape[:-1]), 3).astype(int)

    start_joint_triangles = np.array([[2*len(phi)-1, 3*len(phi)-1,2*len(phi)],[2*len(phi)-1, 2*len(phi),len(phi)]])
    joint_triangles = linrange(start_joint_triangles, len(phi), len(theta)-3)
    joint_faces = joint_triangles.reshape(np.prod(joint_triangles.shape[:-1]), 3).astype(int)

    start_pols_triangles = np.array([[0, len(phi),len(phi)+1],[(len(theta)-2)*len(phi), (len(theta)-1)*len(phi)+1,(len(theta)-2)*len(phi)+1]])
    pols_triangles = linrange(start_pols_triangles, 1, len(phi))
    pols_faces = pols_triangles.reshape(np.prod(pols_triangles.shape[:-1]), 3).astype(int)
    pols_faces[::2,::3] = np.zeros((len(phi),1))
    pols_faces[1::2,1::3] = np.full((len(phi),1), (len(theta)-1)*len(phi))
    pols_faces[-2,-1] = len(phi)
    pols_faces[-1,-1] = len(phi)*(len(theta)-2)
    
    return np.vstack((equator_faces, joint_faces, pols_faces))

Here the `.stl` files are generating with `numpy-stl` package.

In [77]:
# create folder
if not os.path.exists(SH_DIR):
    os.mkdir(SH_DIR)

# generate .stl for spherical harmonics
max_l = 3
# little sphere radius
r = 5  # size in mm
# overall scale
SCALE = 1.5
# number of phi and theta samples
n_phi, n_theta = 500, 500


phi = np.linspace(0, 2*np.pi, n_phi, endpoint=False)
theta = np.linspace(0, np.pi, n_theta)

for l in range(max_l+1):
    for m in range(-l, l+1):
        points = spfunc(phi, theta, m=m, l=l, r=r)
        faces = faces_generator(phi, theta)

        my_mesh = mesh.Mesh(np.zeros(faces.shape[0], dtype=mesh.Mesh.dtype))
        my_mesh.vectors = points[faces,:] * 0.5 * SCALE  # 0.5 somehow works
        my_mesh.save(os.path.join(SH_DIR, f'{l}{abs(m)}{"-" if m < 0 else "+"}.stl'))

# .stl optimization

Define function for mesh optimisation with the `gmsh` package.

In [101]:
import gmsh
import math
import os
import tqdm

# t13 and t17 in gmsh docs https://gmsh.info/doc/texinfo/gmsh.html
def optimize(stl_path, result_path):
    gmsh.clear()
    gmsh.merge(stl_path)

    angle = 120
    forceParametrizablePatches = True
    includeBoundary = True
    curveAngle = 180
    gmsh.model.mesh.classifySurfaces(angle * math.pi / 180., includeBoundary,
                                     forceParametrizablePatches,
                                     curveAngle * math.pi / 180.)
    
    gmsh.model.mesh.createGeometry()
    
    s = gmsh.model.getEntities(2)
    l = gmsh.model.geo.addSurfaceLoop([e[1] for e in s])
    gmsh.model.geo.addVolume([l])

    gmsh.model.geo.synchronize()

    f = gmsh.model.mesh.field.add("MathEval")
    gmsh.model.mesh.field.setString(f, "F", "0.5")
    gmsh.model.mesh.field.setAsBackgroundMesh(f)
    
    # Use bamg
    gmsh.option.setNumber("Mesh.SmoothRatio", 3)
    gmsh.option.setNumber("Mesh.AnisoMax", 1000)
#     gmsh.option.setNumber("Mesh.Algorithm", 7)

    gmsh.model.mesh.generate(2)
    gmsh.write(result_path)

Optimization of all `.stl` files in `SH_DIR` folder.

In [102]:
# create folder
if not os.path.exists(SH_OPT_DIR):
    os.mkdir(SH_OPT_DIR)
    
for file in tqdm.tqdm(os.listdir(SH_DIR)):
    if not os.path.isfile(os.path.join(SH_DIR, file)):
        continue
    try:
        gmsh.initialize()

        optimize(
            stl_path=os.path.join(SH_DIR, file),
            result_path=os.path.join(SH_OPT_DIR, file)
        )
    finally:
        gmsh.finalize()

100%|█████████████████████████████████████████████████████████████████████████████████| 16/16 [33:50<00:00, 126.93s/it]
